In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nazmul0087/ct-kidney-dataset-normal-cyst-tumor-and-stone")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/nazmul0087/ct-kidney-dataset-normal-cyst-tumor-and-stone


In [6]:
"""
CT Kidney (Normal / Cyst / Tumor / Stone) classification
MobileNetV3-Small (last 2 blocks unfrozen) -> dim. reduction -> N_QUBITS-qubit QCNN
(angle enc + quantum conv + MPS + VQC) -> wire cut (N_A-qubit + N_B-qubit fragments)
-> exact reconstruction -> measurement (Z expectations) -> FC layer -> class prediction

Pure PyTorch state-vector simulator (batched, differentiable, GPU-friendly).
Requires: torch, torchvision, kagglehub, scikit-learn

All sizes are derived from N_QUBITS and N_A below, so you can change them freely
(e.g. 8 = 4+5, 10 = 5+6, 12 = 6+7).  Fragment B has one extra qubit: the cut wire.
"""
import os, math, random, copy, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

CDTYPE = torch.complex64

# ----------------------------------------------------------------------------
# Circuit size configuration  (change only N_QUBITS and N_A)
# ----------------------------------------------------------------------------
N_QUBITS = 10                    # total physical qubits of the uncut circuit
N_A = 5                          # qubits in fragment A  (global qubits 0 .. N_A-1)
N_B = N_QUBITS - N_A + 1         # fragment B = cut wire + remaining qubits
CUT_Q_A = N_A - 1                # qubit of fragment A whose wire is cut
assert N_A >= 2 and N_B >= 2, "each fragment needs at least 2 qubits"
assert N_A < N_QUBITS, "N_A must be smaller than N_QUBITS"


# ----------------------------------------------------------------------------
# State-vector simulator
# ----------------------------------------------------------------------------
def _c(m):
    return torch.complex(m, torch.zeros_like(m))


def ry(t):
    c, s = torch.cos(t / 2), torch.sin(t / 2)
    m = torch.stack([torch.stack([c, -s], -1), torch.stack([s, c], -1)], -2)
    return _c(m)


def rz(t):
    a = torch.complex(torch.cos(t / 2), -torch.sin(t / 2))
    z = torch.zeros_like(a)
    return torch.stack([torch.stack([a, z], -1), torch.stack([z, a.conj()], -1)], -2)


_GATES = {}


def _gate(name, device):
    key = (name, str(device))
    if key not in _GATES:
        r = 1 / math.sqrt(2)
        mats = {
            "H": [[r, r], [r, -r]],
            "SDG": [[1, 0], [0, -1j]],
            "CNOT": [[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]],
            "CZ": [[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, -1]],
        }
        _GATES[key] = torch.tensor(mats[name], dtype=CDTYPE, device=device)
    return _GATES[key]


def zero_state(B, n, device):
    s = torch.zeros((B,) + (2,) * n, dtype=CDTYPE, device=device)
    s.view(B, -1)[:, 0] = 1.0
    return s


def apply_1q(state, U, q):
    """U: (2,2) shared or (B,2,2) per-sample."""
    B = state.shape[0]
    s = state.movedim(q + 1, -1)
    shp = s.shape
    s = s.reshape(B, -1, 2)
    if U.dim() == 2:
        out = s @ U.transpose(0, 1)
    else:
        out = torch.einsum("bma,bca->bmc", s, U)
    return out.reshape(shp).movedim(-1, q + 1)


def apply_2q(state, U, q1, q2):
    B = state.shape[0]
    s = state.movedim((q1 + 1, q2 + 1), (-2, -1))
    shp = s.shape
    out = s.reshape(B, -1, 4) @ U.transpose(0, 1)
    return out.reshape(shp).movedim((-2, -1), (q1 + 1, q2 + 1))


def zsigns(n, device):
    idx = torch.arange(2 ** n, device=device)
    shifts = n - 1 - torch.arange(n, device=device)
    bits = (idx[:, None] >> shifts[None, :]) & 1
    return (1 - 2 * bits).float()          # (2^n, n)


def probs_of(state):
    return state.abs().pow(2).reshape(state.shape[0], -1)


# ----------------------------------------------------------------------------
# QCNN fragment: angle encoding -> quantum conv -> MPS -> VQC
# ----------------------------------------------------------------------------
class FragmentParams(nn.Module):
    def __init__(self, m, vqc_layers=2):
        super().__init__()
        self.conv = nn.Parameter(torch.randn(3) * 0.5)             # shared conv filter
        self.mps = nn.Parameter(torch.randn(m - 1, 3) * 0.5)      # MPS staircase
        self.vqc = nn.Parameter(torch.randn(vqc_layers, m, 2) * 0.5)


def _block(state, qa, qb, th):
    dev = state.device
    state = apply_1q(state, ry(th[0]), qa)
    state = apply_1q(state, ry(th[1]), qb)
    state = apply_2q(state, _gate("CNOT", dev), qa, qb)
    state = apply_1q(state, rz(th[2]), qb)
    state = apply_2q(state, _gate("CNOT", dev), qa, qb)
    return state


def apply_fragment(state, qmap, p, enc):
    """
    qmap : list mapping local qubit index -> index in `state`
    enc  : {local_qubit: angles (B,)}  (angle encoding with RY)
    """
    m = len(qmap)
    dev = state.device
    # 1) angle encoding
    for lq, ang in enc.items():
        state = apply_1q(state, ry(ang), qmap[lq])
    # 2) quantum convolution (shared filter, even then odd pairs)
    for start in (0, 1):
        for i in range(start, m - 1, 2):
            state = _block(state, qmap[i], qmap[i + 1], p.conv)
    # 3) MPS-style staircase of two-qubit blocks
    for i in range(m - 1):
        state = _block(state, qmap[i], qmap[i + 1], p.mps[i])
    # 4) VQC: single-qubit rotations + CZ chain
    for l in range(p.vqc.shape[0]):
        for i in range(m):
            state = apply_1q(state, ry(p.vqc[l, i, 0]), qmap[i])
            state = apply_1q(state, rz(p.vqc[l, i, 1]), qmap[i])
        for i in range(m - 1):
            state = apply_2q(state, _gate("CZ", dev), qmap[i], qmap[i + 1])
    return state


# Cut-wire preparation states and Pauli decomposition  rho = 1/2 sum_P Tr(P rho) P
_r = 1 / math.sqrt(2)
PREP = {
    "0": [1, 0], "1": [0, 1],
    "+": [_r, _r], "-": [_r, -_r],
    "+i": [_r, 1j * _r], "-i": [_r, -1j * _r],
}
TERMS = [("I", "0", 1), ("I", "1", 1),
         ("Z", "0", 1), ("Z", "1", -1),
         ("X", "+", 1), ("X", "-", -1),
         ("Y", "+i", 1), ("Y", "-i", -1)]


class CutQCNN(nn.Module):
    """N_QUBITS-qubit QCNN executed as an N_A-qubit and an N_B-qubit circuit (one wire cut)."""

    def __init__(self, vqc_layers=2):
        super().__init__()
        self.pA = FragmentParams(N_A, vqc_layers)
        self.pB = FragmentParams(N_B, vqc_layers)
        # <Z_0 .. Z_{N_QUBITS-1}> + <Z_{N_A-2} Z_{N_A-1}> (correlation across the cut)
        self.n_features = N_QUBITS + 1

    # ---- cut execution + classical reconstruction ----------------------------
    def forward(self, x):
        B, dev = x.shape[0], x.device
        NB_NEIGH = N_A - 2            # qubit of A that neighbours the cut qubit

        # fragment A (global qubits 0 .. N_A-1)
        sA = zero_state(B, N_A, dev)
        sA = apply_fragment(sA, list(range(N_A)), self.pA, {i: x[:, i] for i in range(N_A)})
        zsA = zsigns(N_A, dev)
        zA = probs_of(sA) @ zsA[:, :N_A - 1]                 # <Z_0> .. <Z_{N_A-2}>

        pa, ca = {}, {}
        for P in "IXYZ":
            s = sA
            if P == "X":
                s = apply_1q(s, _gate("H", dev), CUT_Q_A)
            elif P == "Y":
                s = apply_1q(apply_1q(s, _gate("SDG", dev), CUT_Q_A), _gate("H", dev), CUT_Q_A)
            pr = probs_of(s)
            if P == "I":
                pa[P] = torch.ones(B, device=dev)
                ca[P] = pr @ zsA[:, NB_NEIGH]
            else:
                pa[P] = pr @ zsA[:, CUT_Q_A]
                ca[P] = pr @ (zsA[:, NB_NEIGH] * zsA[:, CUT_Q_A])

        # fragment B: one batched run for all 8 (Pauli, eigenstate) terms
        T = len(TERMS)
        init = torch.zeros(T * B, 2, 2 ** (N_B - 1), dtype=CDTYPE, device=dev)
        for t, (_, st, _) in enumerate(TERMS):
            v = torch.tensor(PREP[st], dtype=CDTYPE, device=dev)
            init[t * B:(t + 1) * B, :, 0] = v                # cut wire = v, rest = |0...0>
        init = init.view((T * B,) + (2,) * N_B)
        xb = x[:, N_A:].repeat(T, 1)                         # (T*B, N_B-1)
        sB = apply_fragment(init, list(range(N_B)), self.pB,
                            {i + 1: xb[:, i] for i in range(N_B - 1)})
        zB = (probs_of(sB) @ zsigns(N_B, dev)).reshape(T, B, N_B)

        coef = torch.stack([0.5 * lam * pa[P] for (P, _, lam) in TERMS])   # (T,B)
        coefc = torch.stack([0.5 * lam * ca[P] for (P, _, lam) in TERMS])  # (T,B)
        zBrec = (coef.unsqueeze(-1) * zB).sum(0)                           # (B,N_B): <Z_{N_A-1} .. Z_{N_QUBITS-1}>
        cross = (coefc * zB[..., 0]).sum(0).unsqueeze(1)                   # <Z_{N_A-2} Z_{N_A-1}>
        return torch.cat([zA, zBrec, cross], dim=1)                        # (B, N_QUBITS+1)

    # ---- reference: the same circuit run uncut (for verification) -------------
    @torch.no_grad()
    def forward_uncut(self, x):
        B, dev = x.shape[0], x.device
        s = zero_state(B, N_QUBITS, dev)
        s = apply_fragment(s, list(range(N_A)), self.pA, {i: x[:, i] for i in range(N_A)})
        qmapB = list(range(N_A - 1, N_QUBITS))               # cut wire + remaining qubits
        s = apply_fragment(s, qmapB, self.pB, {i + 1: x[:, N_A + i] for i in range(N_B - 1)})
        zs = zsigns(N_QUBITS, dev)
        pr = probs_of(s)
        cross = pr @ (zs[:, N_A - 2] * zs[:, N_A - 1])
        return torch.cat([pr @ zs, cross.unsqueeze(1)], dim=1)


# ----------------------------------------------------------------------------
# Hybrid model: MobileNetV3-Small (last N blocks trainable) -> reduction -> cut QCNN -> FC
# ----------------------------------------------------------------------------
class HybridKidneyNet(nn.Module):
    def __init__(self, num_classes=4, pretrained=True, unfreeze_last=2):
        super().__init__()
        from torchvision import models
        try:
            w = models.MobileNet_V3_Small_Weights.IMAGENET1K_V1 if pretrained else None
            mb = models.mobilenet_v3_small(weights=w)
        except Exception as e:
            print("[warn] could not load pretrained weights:", e)
            mb = models.mobilenet_v3_small(weights=None)
        self.backbone = mb.features                      # (B,576,7,7) for 224x224 input
        self.pool = nn.AdaptiveAvgPool2d(1)
        feat_dim = 576

        # freeze everything, then unfreeze the last `unfreeze_last` blocks of `features`
        for p in self.backbone.parameters():
            p.requires_grad = False
        self.trainable_blocks = list(self.backbone)[-unfreeze_last:] if unfreeze_last > 0 else []
        for blk in self.trainable_blocks:
            for p in blk.parameters():
                p.requires_grad = True

        # feature dimensionality reduction 576 -> N_QUBITS (one angle per qubit)
        self.reduce = nn.Sequential(
            nn.Linear(feat_dim, 128), nn.ReLU(inplace=True), nn.Dropout(0.3), nn.Linear(128, N_QUBITS))
        self.qcnn = CutQCNN()
        self.norm = nn.BatchNorm1d(self.qcnn.n_features)
        self.fc = nn.Sequential(nn.Linear(self.qcnn.n_features, 32), nn.ReLU(inplace=True),
                                nn.Dropout(0.2), nn.Linear(32, num_classes))

    def train(self, mode=True):
        super().train(mode)
        self.backbone.eval()                             # frozen blocks keep fixed BN statistics
        for blk in self.trainable_blocks:
            blk.train(mode)                              # unfrozen blocks adapt (incl. their BN)
        return self

    def forward(self, img):
        f = self.pool(self.backbone(img)).flatten(1)     # (B,576)
        angles = math.pi * torch.tanh(self.reduce(f))    # (B,N_QUBITS) in (-pi, pi)
        q = self.qcnn(angles)                            # (B,N_QUBITS+1)
        return self.fc(self.norm(q))


# ----------------------------------------------------------------------------
# Data / training utilities
# ----------------------------------------------------------------------------
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")


def find_image_root(path):
    """Return the folder whose sub-folders are the class folders."""
    for dirpath, dirs, _ in os.walk(path):
        good = [d for d in dirs
                if any(f.lower().endswith(IMG_EXT) for f in os.listdir(os.path.join(dirpath, d)))]
        if len(good) >= 2:
            return dirpath
    raise FileNotFoundError("No class folders with images found under " + path)


def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


@torch.no_grad()
def evaluate(model, loader, device, criterion=None):
    model.eval()
    ys, ps, tot_loss, n = [], [], 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        if criterion is not None:
            tot_loss += criterion(out, y).item() * x.size(0)
        n += x.size(0)
        ps.append(out.argmax(1).cpu()); ys.append(y.cpu())
    return tot_loss / max(n, 1), torch.cat(ys).numpy(), torch.cat(ps).numpy()


def main():
    import kagglehub
    from torchvision import datasets, transforms
    from torch.utils.data import DataLoader, Subset
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

    # ---------------- config ----------------
    SEED, IMG, BS, EPOCHS = 42, 224, 32, 15
    UNFREEZE_LAST = 2
    LR_BACKBONE, LR_CLASSICAL, LR_QUANTUM, WD = 1e-4, 1e-3, 2e-2, 1e-4
    set_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)
    print(f"qubits: {N_QUBITS} = fragment A ({N_A}) + fragment B ({N_B}, incl. cut wire)")

    # ---------------- data ----------------
    path = kagglehub.dataset_download("nazmul0087/ct-kidney-dataset-normal-cyst-tumor-and-stone")
    print("Path to dataset files:", path)
    root = find_image_root(path)
    print("image root:", root)

    mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
    tf_train = transforms.Compose([
        transforms.Resize((IMG, IMG)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(0.2, 0.2),
        transforms.ToTensor(), transforms.Normalize(mean, std)])
    tf_eval = transforms.Compose([
        transforms.Resize((IMG, IMG)), transforms.ToTensor(), transforms.Normalize(mean, std)])

    ds_tr = datasets.ImageFolder(root, transform=tf_train)
    ds_ev = datasets.ImageFolder(root, transform=tf_eval)
    classes = ds_tr.classes
    targets = np.array(ds_tr.targets)
    print("classes:", classes, "| counts:", np.bincount(targets))

    idx = np.arange(len(targets))
    tr_idx, tmp_idx = train_test_split(idx, test_size=0.30, stratify=targets, random_state=SEED)
    va_idx, te_idx = train_test_split(tmp_idx, test_size=0.50, stratify=targets[tmp_idx], random_state=SEED)
    print(f"train {len(tr_idx)} | val {len(va_idx)} | test {len(te_idx)}")

    kw = dict(batch_size=BS, num_workers=2, pin_memory=device.type == "cuda")
    train_loader = DataLoader(Subset(ds_tr, tr_idx), shuffle=True, drop_last=True, **kw)
    val_loader = DataLoader(Subset(ds_ev, va_idx), shuffle=False, **kw)
    test_loader = DataLoader(Subset(ds_ev, te_idx), shuffle=False, **kw)

    # ---------------- model ----------------
    model = HybridKidneyNet(num_classes=len(classes), pretrained=True,
                            unfreeze_last=UNFREEZE_LAST).to(device)
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_all = sum(p.numel() for p in model.parameters())
    print(f"trainable params: {n_tr:,} / {n_all:,}")

    # sanity check: the cut reconstruction must equal the uncut circuit
    with torch.no_grad():
        xt = (math.pi * torch.tanh(torch.randn(4, N_QUBITS))).to(device)
        err = (model.qcnn(xt) - model.qcnn.forward_uncut(xt)).abs().max().item()
        print(f"cut vs uncut max abs error: {err:.2e}")

    cnt = np.bincount(targets[tr_idx], minlength=len(classes))
    cw = torch.tensor(cnt.sum() / (len(classes) * cnt), dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=cw)

    bparams = [p for p in model.backbone.parameters() if p.requires_grad]
    qparams = list(model.qcnn.parameters())
    skip = {id(p) for p in bparams} | {id(p) for p in qparams}
    cparams = [p for p in model.parameters() if p.requires_grad and id(p) not in skip]
    optim = torch.optim.AdamW([
        {"params": bparams, "lr": LR_BACKBONE},
        {"params": cparams, "lr": LR_CLASSICAL},
        {"params": qparams, "lr": LR_QUANTUM}], weight_decay=WD)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=EPOCHS)

    # ---------------- train ----------------
    best_f1, best_state = -1.0, None
    for ep in range(1, EPOCHS + 1):
        model.train()
        t0, run, seen, correct = time.time(), 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optim.zero_grad(set_to_none=True)
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 5.0)
            optim.step()
            run += loss.item() * x.size(0); seen += x.size(0)
            correct += (out.argmax(1) == y).sum().item()
        sched.step()
        vl, yv, pv = evaluate(model, val_loader, device, criterion)
        vacc, vf1 = accuracy_score(yv, pv), f1_score(yv, pv, average="macro")
        print(f"ep {ep:02d}/{EPOCHS} | train loss {run/seen:.4f} acc {correct/seen:.4f} | "
              f"val loss {vl:.4f} acc {vacc:.4f} f1 {vf1:.4f} | {time.time()-t0:.0f}s")
        if vf1 > best_f1:
            best_f1, best_state = vf1, copy.deepcopy(model.state_dict())
            torch.save(best_state, "best_kidney_cut_qcnn_mnv3.pt")

    # ---------------- test ----------------
    model.load_state_dict(best_state)
    _, yt, pt = evaluate(model, test_loader, device)
    print("\nTEST accuracy:", round(accuracy_score(yt, pt), 4),
          "| macro-F1:", round(f1_score(yt, pt, average="macro"), 4))
    print(classification_report(yt, pt, target_names=classes, digits=4))
    print("confusion matrix:\n", confusion_matrix(yt, pt))


if __name__ == "__main__":
    main()

device: cuda
qubits: 10 = fragment A (5) + fragment B (6, incl. cut wire)
Path to dataset files: /kaggle/input/datasets/nazmul0087/ct-kidney-dataset-normal-cyst-tumor-and-stone
image root: /kaggle/input/datasets/nazmul0087/ct-kidney-dataset-normal-cyst-tumor-and-stone/CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone/CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone
classes: ['Cyst', 'Normal', 'Stone', 'Tumor'] | counts: [3709 5077 1377 2283]
train 8712 | val 1867 | test 1867
trainable params: 426,305 / 1,002,769
cut vs uncut max abs error: 8.34e-07
ep 01/15 | train loss 0.8019 acc 0.6907 | val loss 0.2731 acc 0.8902 f1 0.8714 | 59s
ep 02/15 | train loss 0.2967 acc 0.8983 | val loss 0.0970 acc 0.9689 f1 0.9602 | 59s
ep 03/15 | train loss 0.2148 acc 0.9284 | val loss 0.0557 acc 0.9754 f1 0.9706 | 59s
ep 04/15 | train loss 0.1503 acc 0.9515 | val loss 0.0688 acc 0.9695 f1 0.9604 | 59s
ep 05/15 | train loss 0.1042 acc 0.9668 | val loss 0.0439 acc 0.9850 f1 0.9781 | 59s
ep 06/15 | train loss 0.1031 acc 0.